In [1]:
import os 
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader

C:\Users\gangt\AppData\Local\Temp\ipykernel_7732\3277348521.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()

True

In [3]:
groq_key= os.getenv("GROQ_API_KEY")
jina_key= os.getenv("JINA_API_KEY")


In [4]:
DATA_FILE_PATH = os.path.join( "data", "hr_policy.txt")

In [5]:
loader= TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents=loader.load()

print("data loaded ")
print("="*40)
print(documents)
len(documents)

data loaded 
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nD

1

In [6]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [7]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks=splitter.split_documents(documents)
len(chunks)
print(chunks[1].page_content)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.


In [11]:
from langchain_community.embeddings import JinaEmbeddings

embedding_model=JinaEmbeddings( model="jina-embeddings-v2-base-en")

In [12]:
from langchain_community.vectorstores import FAISS 

vector_store= FAISS.from_documents(chunks, embedding_model)

print("chunks are stored ", vector_store.index.ntotal)

RuntimeError: Insufficient account balance. Top up your account at https://jina.ai/api-dashboard/key-manager.

In [ ]:
test_query="how many sick  leaves does employee get"

top_matches=vector_store.similarity_search(test_query, k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches, start=1):
    print(f"---match ---{i}")
    print(match.page_content)
    print()

RuntimeError: Insufficient account balance. Top up your account at https://jina.ai/api-dashboard/key-manager.

In [16]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-120b",
             temperature=0.9)

print(llm.model_name)

openai/gpt-oss-120b


In [17]:
print(llm.invoke("learning Rag is hard ? "))

content='**Short answer:**\u202fYes, many people find learning Retrieval‑Augmented Generation (RAG) challenging at first, but with a clear roadmap and the right resources you can master it step by step.\n\n---\n\n## 1️⃣ Why RAG feels hard at first\n\n| Aspect | Why it trips people up | How to tame it |\n|--------|-----------------------|----------------|\n| **Conceptual blend** – combines information retrieval, dense embeddings, and LLM prompting. | You have to think in two “worlds” (search + generation) simultaneously. | Treat each component as its own mini‑project first (e.g., build a simple vector store, then a prompt template). |\n| **Tooling variety** – Faiss, Milvus, Chroma, Pinecone, LangChain, LlamaIndex, etc. | Too many libraries → choice paralysis. | Pick **one** stack (e.g., LangChain\u202f+\u202fFAISS) and stick with it for the first 2–3 projects. |\n| **Data hygiene** – Chunking, cleaning, metadata design. | Small mistakes (over‑large chunks, missing IDs) break retrieval. 

In [ ]:
from langchain.agents import create_agent

hr_assistant= create_agent(
    model=llm,
    tools=[],
    system_prompt="""
        you are a friendly hr assistant , 
        always use the search_hr_policy tool to look up 
        facts before answering .
        if the answer isn't in the search results , say you dont now 
        instead of guessing 
"""
)